# 08 — Historia y diversidad de activos

> ¿Cómo cambia la utilidad económica de una LSTM frente a una red feedforward al ampliar la historia y la diversidad económica del entrenamiento?

La evaluación de transferencia sobre XLF entre 2024-01-02 y 2026-11-17 es exploratoria: tanto el activo como esos años ya fueron examinados. No se selecciona una configuración ganadora.

## <b>Block 01 — Dataset congelado y baseline causal</b>

> Se leen únicamente las tres matrices de `dmn_historical_2010_v1`. Los hashes se verifican antes de cargar precios; BIL, staging y el Gold previo quedan fuera.

In [ ]:
from pathlib import Path
import hashlib, json, math, random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import torch
from torch import nn
torch.set_num_threads(1)

SEED,EPOCHS,LR=42,80,.01
LOOKBACK,LAMBDA,WARMUP,TARGET,LIMIT,COST=63,.94,20,.12,.20,.0005
EVAL_START,EVAL_END=pd.Timestamp('2024-01-02'),pd.Timestamp('2026-11-17')
XLF='XLF'
CONDITIONS={
 'A · 2018+ / equity styles':(pd.Timestamp('2018-01-02'),['SPY','QQQ','IWM','IWD','IWF','XLE']),
 'B · 2010+ / equity styles':(pd.Timestamp('2010-01-04'),['SPY','QQQ','IWM','IWD','IWF','XLE']),
 'C · 2018+ / diversified':(pd.Timestamp('2018-01-02'),['SPY','QQQ','XLE','GLD','TLT','SHY']),
 'D · 2010+ / diversified':(pd.Timestamp('2010-01-04'),['SPY','QQQ','XLE','GLD','TLT','SHY']),}
CYCLES=[('2024',pd.Timestamp('2023-12-31')),('2025',pd.Timestamp('2024-12-31')),('2026',pd.Timestamp('2025-12-31'))]

def repo_root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'Momentum_systematic_strategies').is_dir(): return p
    raise FileNotFoundError('No se encontró la raíz del repositorio.')
ROOT=repo_root(); DATA=ROOT/'Momentum_systematic_strategies/data/gold_versions/dmn_historical_2010_v1'
freeze=json.loads((DATA/'FREEZE_MANIFEST.json').read_text(encoding='utf-8'))
assert freeze['dataset_version']=='dmn_historical_2010_v1' and 'BIL' in freeze['excluded']
for name in ('main_adjusted_close.parquet','main_simple_returns.parquet','main_log_returns.parquet'):
    assert hashlib.sha256((DATA/name).read_bytes()).hexdigest()==freeze['sha256'][name], name
raw=pl.read_parquet(DATA/'main_adjusted_close.parquet').sort('session_date').to_dict(as_series=False)
prices=pd.DataFrame(raw).rename(columns={'session_date':'date'}); prices['date']=pd.to_datetime(prices['date'])
assert {'BIL','conditional_bil_adjusted_close'}.isdisjoint(prices.columns)
print(f"Panel congelado verificado: {prices.date.min().date()} → {prices.date.max().date()}, {len(prices)} sesiones.")
print('Hashes de matrices: OK; XLF se reserva solo para inferencia y evaluación.')

## <b>Block 02 — Diseño factorial y modelos compartidos</b>

> A/B separan el efecto de historia; A/C y B/D cambian la composición económica. Las seis series de cada condición son las únicas que entran en normalización, entrenamiento y gradientes. La señal en $t$ se aplica al retorno de $t+1$.

In [ ]:
FEATURES=['ret_1d','mom_21','mom_63','mom_126','ewma_vol_annual']
def make_frame(symbol,start):
    f=prices.loc[prices.date>=start,['date',symbol]].rename(columns={symbol:'price'}).copy(); f['ret_1d']=f.price.pct_change()
    for h in (21,63,126): f[f'mom_{h}']=np.log(f.price/f.price.shift(h))
    r=f.ret_1d.to_numpy(float); v=np.full(len(f),np.nan); first=np.flatnonzero(np.isfinite(r))[0]; init=first+WARMUP-1; v[init]=np.mean(r[first:init+1]**2)
    for i in range(init+1,len(f)): v[i]=LAMBDA*v[i-1]+(1-LAMBDA)*r[i]**2
    f['ewma_vol_annual']=np.sqrt(v*252); f['baseline']=np.where(f.mom_126>0,np.minimum(1.,TARGET/f.ewma_vol_annual),0.)
    f['return_date']=f.date.shift(-1); f['next_return']=f.ret_1d.shift(-1); f['eligible']=f[FEATURES].notna().all(axis=1).rolling(LOOKBACK).sum().eq(LOOKBACK) & f.next_return.notna()
    first_eligible=np.flatnonzero(f.eligible.to_numpy())[0]; f=f.iloc[first_eligible-LOOKBACK+1:].reset_index(drop=True); f['row']=np.arange(len(f)); return f

class SharedFF(nn.Module):
    def __init__(self):
        super().__init__(); self.hidden=nn.Linear(5,8); self.residual=nn.Linear(8,1); nn.init.zeros_(self.residual.weight); nn.init.zeros_(self.residual.bias)
    def forward(self,x): return LIMIT*torch.tanh(self.residual(torch.tanh(self.hidden(x))).squeeze(-1))
class SharedLSTM(nn.Module):
    def __init__(self):
        super().__init__(); self.lstm=nn.LSTM(5,8,batch_first=True); self.residual=nn.Linear(8,1); nn.init.zeros_(self.residual.weight); nn.init.zeros_(self.residual.bias)
    def forward(self,x): return LIMIT*torch.tanh(self.residual(self.lstm(x)[0][:,-1,:]).squeeze(-1))
def pred(model,x):
    a,t=x.shape[:2]; return model(x.reshape(a*t,*x.shape[2:])).reshape(a,t)
def fit(model,x,base,rets,index):
    opt=torch.optim.Adam(model.parameters(),lr=LR); ix=torch.tensor(index,dtype=torch.long)
    for _ in range(EPOCHS):
        opt.zero_grad(); p=torch.clamp(base[:,ix]+pred(model,x[:,ix]),0,1); turn=(p[:,1:]-p[:,:-1]).abs(); portfolio=(p[:,1:]*rets[:,ix][:,1:]-COST*turn).mean(0); loss=-(portfolio.mean()/portfolio.std(unbiased=False).clamp_min(1e-6)*math.sqrt(252)); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step()
    return model
print(pd.DataFrame({'modelo':['TSMOM','Feedforward compartida','LSTM compartida'],'parámetros entrenables':[0,57,489]}))

## <b>Block 03 — Walk-forward y continuidad económica</b>

> Cada ciclo se reinicializa con semilla 42 y sólo entrena con retornos disponibles al cierre del año anterior. Las sesiones entre ciclos mantienen la posición heredada sin rebalanceo ni coste ficticio; 2024 comienza en efectivo.

In [ ]:
def run_condition(label,start,assets):
    frames={s:make_frame(s,start) for s in assets+[XLF]}; common=set.intersection(*(set(frames[s].date) for s in assets+[XLF])); common=sorted(common)
    frames={s:frames[s].loc[frames[s].date.isin(common)].reset_index(drop=True) for s in frames}; assert all(np.array_equal(frames[s].date,frames[XLF].date) for s in frames)
    all_dates=frames[XLF].date.to_numpy(); parts=[]; prev={k:0. for k in ['TSMOM','Feedforward','LSTM']}; schedule=[]
    for year,train_end in CYCLES:
        train_idx=np.flatnonzero((frames[XLF].eligible&(frames[XLF].return_date<=train_end)).to_numpy()); raw_train=np.concatenate([frames[s].loc[train_idx,FEATURES].to_numpy(np.float32) for s in assets]); mean=raw_train.mean(0); std=np.maximum(raw_train.std(0),1e-8)
        seq=[]; cur=[]; base=[]; rets=[]
        for s in assets+[XLF]:
            full=(frames[s][FEATURES].to_numpy(np.float32)-mean)/std; padded=np.vstack((np.repeat(full[:1],LOOKBACK-1,axis=0),full)); seq.append(np.stack([padded[i:i+LOOKBACK] for i in frames[s].row])); cur.append(full[frames[s].row]); base.append(frames[s].baseline.to_numpy(np.float32)); rets.append(frames[s].next_return.to_numpy(np.float32))
        xseq,xcur,b,r=torch.tensor(np.stack(seq[:-1])),torch.tensor(np.stack(cur[:-1])),torch.tensor(np.stack(base[:-1])),torch.tensor(np.stack(rets[:-1]))
        np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED); lstm=fit(SharedLSTM(),xseq,b,r,train_idx); ff=fit(SharedFF(),xcur,b,r,train_idx)
        with torch.no_grad(): positions={'TSMOM':base[-1],'Feedforward':np.clip(base[-1]+pred(ff,torch.tensor(cur[-1][None])).numpy().ravel(),0,1),'LSTM':np.clip(base[-1]+pred(lstm,torch.tensor(seq[-1][None])).numpy().ravel(),0,1)}
        target=np.flatnonzero((frames[XLF].eligible&(frames[XLF].date>=max(EVAL_START,pd.Timestamp(f'{year}-01-01')))&(frames[XLF].date<=min(EVAL_END,pd.Timestamp(f'{year}-12-31')))).to_numpy()); available=frames[XLF].loc[train_idx,'return_date'].max(); bridge=target[frames[XLF].loc[target,'date'].to_numpy()<=available]; active=target[frames[XLF].loc[target,'date'].to_numpy()>available]
        for strategy,p in positions.items():
            if len(bridge): parts.append(pd.DataFrame({'condition':label,'history_start':start.date(),'universe':'/'.join(assets),'cycle':year,'strategy':strategy,'decision_date':frames[XLF].loc[bridge,'date'].to_numpy(),'return_date':frames[XLF].loc[bridge,'return_date'].to_numpy(),'position':prev[strategy],'gross':prev[strategy]*rets[-1][bridge],'costs':0.,'turnover':0.}))
            w=p[active]; turn=np.r_[abs(w[0]-prev[strategy]),abs(np.diff(w))]; parts.append(pd.DataFrame({'condition':label,'history_start':start.date(),'universe':'/'.join(assets),'cycle':year,'strategy':strategy,'decision_date':frames[XLF].loc[active,'date'].to_numpy(),'return_date':frames[XLF].loc[active,'return_date'].to_numpy(),'position':w,'gross':w*rets[-1][active],'costs':COST*turn,'turnover':turn})); prev[strategy]=w[-1]
        schedule.append({'condition':label,'cycle':year,'train decisions':len(train_idx),'first new decision':frames[XLF].loc[active,'date'].min().date(),'bridge sessions':len(bridge)})
    out=pd.concat(parts).sort_values(['strategy','decision_date']).reset_index(drop=True); out['net']=out.gross-out.costs
    assert np.allclose(out.net,out.gross-out.costs) and out.position.between(0,1).all()
    expected=frames[XLF].loc[frames[XLF].eligible&(frames[XLF].date>=EVAL_START)&(frames[XLF].date<=EVAL_END),'date']; assert out.groupby('strategy').decision_date.apply(tuple).nunique()==1 and pd.DatetimeIndex(out.loc[out.strategy=='TSMOM','decision_date']).equals(pd.DatetimeIndex(expected))
    return out,pd.DataFrame(schedule)

daily_parts=[]; schedules=[]
for label,(start,assets) in CONDITIONS.items():
    d,s=run_condition(label,start,assets); daily_parts.append(d); schedules.append(s)
daily=pd.concat(daily_parts).reset_index(drop=True); schedule=pd.concat(schedules).reset_index(drop=True)
print('Observaciones por condición/estrategia:',daily.groupby(['condition','strategy']).size().unique())
display(schedule)

## <b>Block 04 — Resultados y atribución descriptiva</b>

> La separación entre exposición, retorno bruto y costes evita interpretar automáticamente una diferencia de NAV como evidencia de memoria recurrente. Las comparaciones son descriptivas y no validan una estrategia.

In [ ]:
def metrics(x):
    n=x.net.to_numpy(); nav=np.r_[1.,np.cumprod(1+n)]; sd=n.std(ddof=0); return pd.Series({'retorno neto compuesto (%)':(nav[-1]-1)*100,'Sharpe neto':n.mean()/max(sd,1e-8)*math.sqrt(252),'volatilidad anual (%)':sd*math.sqrt(252)*100,'max drawdown (%)':(nav/np.maximum.accumulate(nav)-1).min()*100,'exposición media (%)':x.position.mean()*100,'turnover total':x.turnover.sum(),'costes totales (%)':x.costs.sum()*100})
summary=daily.groupby(['condition','strategy'],sort=False).apply(metrics,include_groups=False).reset_index(); annual=daily.groupby(['condition','cycle','strategy'],sort=False).apply(metrics,include_groups=False).reset_index()
base=daily.loc[daily.strategy=='TSMOM',['condition','decision_date','net','gross','costs','position']].rename(columns=lambda x:'base_'+x if x not in ['condition','decision_date'] else x)
compare=daily.loc[daily.strategy!='TSMOM'].merge(base,on=['condition','decision_date']); compare['Δ neto diario vs TSMOM (bps)']=(compare.net-compare.base_net)*1e4; compare['Δ bruto (%)']=(compare.gross-compare.base_gross)*100; compare['Δ costes (%)']=(compare.costs-compare.base_costs)*100; compare['Δ exposición']=(compare.position-compare.base_position)
attr=compare.groupby(['condition','strategy']).agg(**{'Δ neto diario vs TSMOM (bps)':('Δ neto diario vs TSMOM (bps)','mean'),'Δ bruto suma (%)':('Δ bruto (%)','sum'),'Δ costes suma (%)':('Δ costes (%)','sum'),'Δ exposición media':('Δ exposición','mean')}).reset_index()
pivot=summary.pivot(index='condition',columns='strategy',values=['retorno neto compuesto (%)','Sharpe neto']).round(3); interaction=summary.pivot(index='condition',columns='strategy',values='Sharpe neto').assign(**{'LSTM − Feedforward':lambda x:x['LSTM']-x['Feedforward']}).reset_index()
display(summary.round(3)); display(annual[['condition','cycle','strategy','retorno neto compuesto (%)','Sharpe neto','costes totales (%)']].round(3)); display(attr.round(3)); display(interaction.round(3))
fig,axes=plt.subplots(2,2,figsize=(12,7),sharex=True,layout='constrained')
for ax,(condition,p) in zip(axes.ravel(),daily.groupby('condition',sort=False)):
    for strategy,q in p.groupby('strategy',sort=False):
        q=q.sort_values('decision_date'); ax.plot(q.decision_date,np.cumprod(1+q.net),label=strategy,lw=1.15)
    ax.set_title(condition); ax.set_ylabel('crecimiento de 1'); ax.legend(fontsize=8)
for ax in axes[-1]: ax.set_xlabel('fecha de decisión')
plt.show()
# Diagnóstico compacto de exposición y contribución económica.
display(compare.groupby(['condition','strategy']).agg(exposición_media=('position','mean'),**{'Δ bruto suma (%)':('Δ bruto (%)','sum'),'Δ costes suma (%)':('Δ costes (%)','sum'),'Δ neto suma (%)':('Δ neto diario vs TSMOM (bps)',lambda x:x.sum()/100)}).round(4))

## <b>Conclusión</b>

- El diseño separa descriptivamente historia, composición económica e interacción, sin usar XLF para entrenamiento ni normalización.
- Feedforward y LSTM comparten variables, coste, límite residual y protocolo temporal; sus diferencias no aíslan por completo la “memoria” porque las capacidades arquitectónicas no son idénticas.
- La atribución reporta exposición, retorno bruto y costes para evitar atribuir todo diferencial a una supuesta habilidad predictiva.
- El periodo 2024–2026 y XLF ya fueron inspeccionados: los resultados son exploratorios y no constituyen validación prospectiva ni evidencia de alpha.